In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")


2025-02-06 13:35:21.579813: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738848921.590998 3131695 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738848921.594314 3131695 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-06 13:35:21.606996: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1738848922.915312 3131695 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1115 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'LPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] # 10 modes for now

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [6]:
n_layers = [2,4,6,8]
units = [32,65,128,256]
loss_func = ['MSE', 'MAE', 'WMSE']
Nepochs = 100
learning_rate = 0.001
arch_df = pd.DataFrame(product(n_layers,units,loss_func))

In [ ]:
for i in arch_df.index[0:2]:
    ######## define architecture:
    model_name = 'simple_network'
    n_dense_layers = arch_df.loc[i, 0] #number of dense layers
    dense_layer_units = arch_df.loc[i, 1] #neurons per dense layer
    loss_func = arch_df.loc[i, 2]

    ###### Checkpointing

    checkpoint_dir = f'./checkpoints/short-run-tests/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}.model.keras'

    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1)

    ######## map out model architecture
    #### input layer
    nn_input = keras.Input(shape=(len(inputs),))

    #### dense layer(s)
    for n_dense_layer in range(n_dense_layers):
        if n_dense_layer == 0:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
        else:
            dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

    #### output layer
    nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

    ######## store architecture as keras model
    model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

    tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'../logs/short-run-tests/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}')

    model.compile(loss='MSE', optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

    history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=10000, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback]) 

Epoch 1/100
676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0339
Epoch 1: saving model to ./checkpoints/short-run-tests/chk-simple_network-nlayers-2-nunits-32-epochs-100-lrate-0.001.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0339 - val_loss: 0.0153
Epoch 2/100
676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0151
Epoch 2: saving model to ./checkpoints/short-run-tests/chk-simple_network-nlayers-2-nunits-32-epochs-100-lrate-0.001.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0151 - val_loss: 0.0148
Epoch 3/100
676/676 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0147
Epoch 3: saving model to ./checkpoints/short-run-tests/chk-simple_network-nlayers-2-nunits-32-epochs-100-lrate-0.001.model.keras
676/676 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0147 - val_loss: 0.0146
Epoch 4/100
655/676 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0145
Epoch 4: saving model to ./checkpoints/short-run-tests/chk-simple_network-nlayers-2-nunits-32-epochs-100-lrate-0.001